# Agent 2 — Data Ingestion

**What this notebook does:**  
Loads all four provided CSV files, downloads stock prices from Yahoo Finance for all 50+ companies, and merges everything into one master table.  
At the end you will have a single spreadsheet with every company's identifiers, ESG data, governance data, EU Taxonomy data, and price history — all in one place.

**How to present this to investors:**  
> *Our data ingestion agent combines four structured datasets provided by the course with live market data from Yahoo Finance, creating a single verified master dataset covering 50+ European companies. Every data point is tagged with a vintage date so the panel can see exactly when the data was collected.*

## Step 1 — Load the four provided data files

**Before running:** Make sure you have copied the four CSV files into the `data/provided/` folder:
- `equityBicsV2.csv`
- `esgEnvironmentalSocialConsolidatedV4.csv`
- `esgGovernanceConsolidatedV4.csv`
- `legalEntityEuTaxonomy.csv`

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import date

TODAY = str(date.today())  # records when we ran this
DATA_DIR = "../data/provided"
MARKET_DIR = "../data/market"

# Load identifiers (this is the 'spine' — all other files join to this one)
df_ids = pd.read_csv(f"{DATA_DIR}/equityBicsV2.csv")
print(f"Identifiers loaded: {len(df_ids)} companies")
df_ids.head(3)

Identifiers loaded: 56 companies


,companyName,ticker,isin,country,bics_sector,bics_industry,bics_sub_industry,bics_activity,market_cap_eur_bn,index_weight_pct
0,ASML Holding,ASML.AS,NL0010273215,Netherlands,Technology,Semiconductors,Semiconductor Equipment,Lithography Equipment,134.2,0.401
1,SAP SE,SAP.DE,DE0007164600,Germany,Technology,Software,Enterprise Software,ERP Systems,333.0,0.766
2,Capgemini,CAP.PA,FR0000125338,France,Technology,IT Services,Consulting,Digital Transformation,257.5,0.254


In [2]:
# Show the column names so we know what we're working with
print("Columns in identifiers file:")
print(list(df_ids.columns))

Columns in identifiers file:
['companyName', 'ticker', 'isin', 'country', 'bics_sector', 'bics_industry', 'bics_sub_industry', 'bics_activity', 'market_cap_eur_bn', 'index_weight_pct']


In [3]:
# Load ESG environmental + social data
df_esg = pd.read_csv(f"{DATA_DIR}/esgEnvironmentalSocialConsolidatedV4.csv")
print(f"ESG E+S data loaded: {len(df_esg)} rows, {len(df_esg.columns)} columns")
print("Columns:", list(df_esg.columns)[:10], "...")

ESG E+S data loaded: 56 rows, 28 columns
Columns: ['isin', 'reporting_year', 'scope1_emissions_tco2e', 'scope2_emissions_market_tco2e', 'scope2_emissions_location_tco2e', 'scope3_emissions_tco2e', 'scope3_categories_reported', 'total_energy_consumption_gwh', 'renewable_energy_pct', 'energy_intensity_mwh_per_eur_m_revenue'] ...


In [4]:
# Load governance data
df_gov = pd.read_csv(f"{DATA_DIR}/esgGovernanceConsolidatedV4.csv")
print(f"Governance data loaded: {len(df_gov)} rows, {len(df_gov.columns)} columns")
print("Columns:", list(df_gov.columns)[:10], "...")

Governance data loaded: 56 rows, 24 columns
Columns: ['isin', 'reporting_year', 'board_size', 'board_independence_pct', 'women_on_board_pct', 'ceo_chair_separation', 'ceo_tenure_years', 'audit_committee_independence_pct', 'remuneration_committee_independence_pct', 'ceo_pay_ratio'] ...


In [5]:
# Load EU Taxonomy data
df_tax = pd.read_csv(f"{DATA_DIR}/legalEntityEuTaxonomy.csv")
print(f"EU Taxonomy data loaded: {len(df_tax)} rows, {len(df_tax.columns)} columns")
print("Columns:", list(df_tax.columns)[:10], "...")

EU Taxonomy data loaded: 56 rows, 24 columns
Columns: ['isin', 'reporting_year', 'taxonomy_eligible_turnover_pct', 'taxonomy_aligned_turnover_pct', 'taxonomy_eligible_capex_pct', 'taxonomy_aligned_capex_pct', 'taxonomy_eligible_opex_pct', 'taxonomy_aligned_opex_pct', 'climate_mitigation_eligible', 'climate_adaptation_eligible'] ...


## Step 2 — Find the common identifier column

All four files share a company identifier (ISIN or ticker). This cell finds and confirms which column to use for joining.

In [6]:
# Show first few rows of each to find the join key
print("=== Identifiers (first 2 rows) ===")
print(df_ids.head(2).to_string())
print("\n=== ESG E+S (first 2 rows) ===")
print(df_esg.head(2).to_string())

=== Identifiers (first 2 rows) ===
    companyName   ticker          isin      country bics_sector   bics_industry        bics_sub_industry          bics_activity  market_cap_eur_bn  index_weight_pct
0  ASML Holding  ASML.AS  NL0010273215  Netherlands  Technology  Semiconductors  Semiconductor Equipment  Lithography Equipment              134.2             0.401
1        SAP SE   SAP.DE  DE0007164600      Germany  Technology        Software      Enterprise Software            ERP Systems              333.0             0.766

=== ESG E+S (first 2 rows) ===
           isin  reporting_year  scope1_emissions_tco2e  scope2_emissions_market_tco2e  scope2_emissions_location_tco2e  scope3_emissions_tco2e  scope3_categories_reported  total_energy_consumption_gwh  renewable_energy_pct  energy_intensity_mwh_per_eur_m_revenue  water_withdrawal_m3  water_consumption_m3  water_stress_area_pct  waste_generated_tonnes  waste_recycled_pct  hazardous_waste_tonnes  employees_total  women_in_workforce_pct

In [7]:
# === ACTION REQUIRED ===
# After running the cell above, look at the output and set JOIN_KEY
# to the column name that appears in ALL four files (likely 'isin' or 'ticker').

JOIN_KEY = "isin"  # <-- change this if the column has a different name

print(f"Using '{JOIN_KEY}' as the join key")
print(f"Sample values: {df_ids[JOIN_KEY].head(5).tolist()}")

Using 'isin' as the join key
Sample values: ['NL0010273215', 'DE0007164600', 'FR0000125338', 'ES0109067019', 'SE0015961909']


## Step 3 — Merge all four files into one master table

In [8]:
# Merge step by step (like combining spreadsheets using a common column)
master = df_ids.copy()

master = master.merge(df_esg, on=JOIN_KEY, how="left", suffixes=("", "_esg"))
print(f"After adding ESG E+S: {master.shape}")

master = master.merge(df_gov, on=JOIN_KEY, how="left", suffixes=("", "_gov"))
print(f"After adding Governance: {master.shape}")

master = master.merge(df_tax, on=JOIN_KEY, how="left", suffixes=("", "_tax"))
print(f"After adding EU Taxonomy: {master.shape}")

# Tag with data vintage
master["data_vintage"] = TODAY

print(f"\nMaster table: {len(master)} companies x {len(master.columns)} columns")
master.head(3)

After adding ESG E+S: (58, 37)
After adding Governance: (62, 60)
After adding EU Taxonomy: (70, 83)

Master table: 70 companies x 84 columns


,companyName,ticker,isin,country,bics_sector,bics_industry,bics_sub_industry,bics_activity,market_cap_eur_bn,index_weight_pct,...,dnsh_climate_adaptation,dnsh_water,dnsh_circular_economy,dnsh_pollution,dnsh_biodiversity,minimum_social_safeguards,green_revenue_proxy_pct,fossil_fuel_revenue_pct,taxonomy_disclosure_quality,data_vintage
0,ASML Holding,ASML.AS,NL0010273215,Netherlands,Technology,Semiconductors,Semiconductor Equipment,Lithography Equipment,134.2,0.401,...,NaN,Met,Met,NaN,NaN,Met,23.6,0.6,Partial,2026-05-06
1,SAP SE,SAP.DE,DE0007164600,Germany,Technology,Software,Enterprise Software,ERP Systems,333.0,0.766,...,Met,NaN,Met,NaN,Met,Met,19.2,1.4,Minimal,2026-05-06
2,Capgemini,CAP.PA,FR0000125338,France,Technology,IT Services,Consulting,Digital Transformation,257.5,0.254,...,NaN,NaN,NaN,NaN,NaN,Met,30.8,1.2,Full,2026-05-06


## Step 4 — Download market prices from Yahoo Finance

This fetches 5 years of historical stock prices. It may take 2–3 minutes.  
The data is saved locally so you don't have to re-download it every time.

In [9]:
import yfinance as yf

# === ACTION REQUIRED ===
# Set the name of the column in equityBicsV2.csv that contains Yahoo Finance tickers
TICKER_COLUMN = "ticker"  # <-- change if the column is named differently

tickers = master[TICKER_COLUMN].dropna().tolist()
print(f"Downloading prices for {len(tickers)} tickers...")
print("Sample tickers:", tickers[:5])

Sample tickers: ['ASML.AS', 'SAP.DE', 'CAP.PA', 'AMS.MC', 'HEXA-B.ST']


In [10]:
price_file = f"{MARKET_DIR}/prices_{TODAY}.csv"

if os.path.exists(price_file):
    print(f"Loading cached prices from {price_file}")
    prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
else:
    print("Downloading from Yahoo Finance (takes ~2 min)...")
    raw = yf.download(tickers, start="2020-01-01", auto_adjust=True, progress=True)
    prices = raw["Close"]
    prices.to_csv(price_file)
    print(f"Saved to {price_file}")

print(f"\nPrice data: {prices.shape[0]} trading days x {prices.shape[1]} stocks")
prices.tail(3)

Loading cached prices from ../data/market/prices_2026-05-06.csv

Price data: 1306 trading days x 56 stocks


,ASML.AS,SAP.DE,CAP.PA,AMS.MC,HEXA-B.ST,SIE.DE,ABBN.SW,SU.PA,AIR.PA,RR.L,...,VOW3.DE,BMW.DE,STLAM.MI,VOLV-B.ST,RNO.PA,DTE.DE,ORA.PA,VOD.L,VNA.DE,GFC.PA
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-30,2503.944047,191.879286,539.260508,334.327630,849.011787,169.285042,233.802235,244.717594,240.526716,115.848527,...,465.123514,169.612487,199.906349,356.462506,72.711032,69.545250,291.829679,278.467231,135.413585,208.005428
2024-12-31,2498.380889,193.745418,538.454075,334.298857,859.631534,165.896413,234.487927,242.584644,239.041194,115.432227,...,461.812106,167.161282,198.310260,355.095119,73.669727,70.119349,296.739103,279.479670,133.889458,208.064887
2025-01-01,2484.123574,187.886952,540.806667,336.914491,860.659779,163.221607,232.173413,245.588630,240.454045,115.540756,...,472.148116,165.860185,200.917655,360.855243,74.712689,69.419358,292.719921,282.369391,131.954815,211.355452


## Step 5 — Data quality check

Before using any data, we check how much is missing. This matters for the investor panel — they will ask about data gaps.

In [11]:
# What percentage of each column is missing?
missing = master.isnull().mean().mul(100).round(1)
missing_summary = missing[missing > 0].sort_values(ascending=False)

print(f"Columns with missing data ({len(missing_summary)} out of {len(master.columns)}):")
print(missing_summary.to_string())

Columns with missing data (19 out of 84):
dnsh_climate_adaptation          61.4
taxonomy_aligned_capex_pct       60.0
dnsh_biodiversity                58.6
dnsh_water                       55.7
dnsh_pollution                   52.9
dnsh_circular_economy            48.6
taxonomy_aligned_turnover_pct    35.7
taxonomy_aligned_opex_pct        35.7
dnsh_climate_mitigation          35.7
net_zero_target_year             30.0
water_consumption_m3             24.3
water_withdrawal_m3              24.3
executive_pay_esg_linked_pct     22.9
scope3_emissions_tco2e           15.7
minimum_social_safeguards        15.7
external_esg_assurance           12.9
lobbying_disclosure               8.6
ceo_pay_ratio                     8.6
assurance_provider                5.7


In [12]:
# How many stocks have prices vs. missing prices?
downloaded = prices.columns.tolist()
all_tickers = master[TICKER_COLUMN].dropna().tolist()
missing_prices = [t for t in all_tickers if t not in downloaded]

print(f"Stocks with price data: {len(downloaded)}")
print(f"Stocks missing prices:  {len(missing_prices)}")
if missing_prices:
    print(f"Missing: {missing_prices}")
    print("(These tickers may need to be corrected — check Yahoo Finance manually)")

Stocks with price data: 56
Stocks missing prices:  0


## Step 6 — Save the master dataset

In [13]:
output_path = f"../outputs/scores/master_dataset_{TODAY}.csv"
master.to_csv(output_path, index=False)
print(f"Master dataset saved to: {output_path}")
print(f"Shape: {master.shape[0]} companies x {master.shape[1]} columns")
print(f"Data vintage: {TODAY}")

Master dataset saved to: ../outputs/scores/master_dataset_2026-05-06.csv
Shape: 70 companies x 84 columns
Data vintage: 2026-05-06


## ✅ Notebook complete

You now have:
- A **master dataset** with all ESG, governance, and taxonomy data merged
- **5 years of stock prices** cached locally
- A **data quality report** showing which fields have missing values

**Next:** Open `02_financial_analysis.ipynb` to calculate returns, volatility, and Sharpe ratios.